# Big Mart Sales Prediction

Regression project using the supplied Train.csv and Test.csv datasets.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [ ]:
ROOT = Path('..') if Path('../data/Train.csv').exists() else Path('.')
train = pd.read_csv(ROOT/'data'/'Train.csv')
test = pd.read_csv(ROOT/'data'/'Test.csv')
print(train.shape, test.shape)
train.head()


## Data quality and target exploration

In [ ]:
train.info()
train.isna().sum()


In [ ]:
train['Item_Outlet_Sales'].describe()


## Preprocessing and model training

The preprocessing is fitted only on training folds to avoid leakage. Median imputation is used for numeric features and most-frequent imputation plus one-hot encoding for categorical features.

In [ ]:
X = train.drop(columns='Item_Outlet_Sales'); y = train['Item_Outlet_Sales']
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()
pre = ColumnTransformer([('num', SimpleImputer(strategy='median'), num), ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)])
Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=.2, random_state=42)
Xtr_t = pre.fit_transform(Xtr); Xv_t = pre.transform(Xv)
model = HistGradientBoostingRegressor(max_iter=180, learning_rate=.08, l2_regularization=1, random_state=42)
model.fit(Xtr_t, ytr)
pred = model.predict(Xv_t)
print('R2:', r2_score(yv,pred)); print('MAE:',mean_absolute_error(yv,pred)); print('RMSE:',mean_squared_error(yv,pred)**.5)


## Test-file prediction

In [ ]:
all_X_t = pre.fit_transform(X); test_t = pre.transform(test)
model.fit(all_X_t,y)
test_pred = model.predict(test_t)
pd.DataFrame({'Item_Outlet_Sales_Predicted':test_pred}).to_csv(ROOT/'results'/'Test_Predictions.csv',index=False)
